In [ ]:
pip install transformers datasets sentencepiece evaluate accelerate rouge_score

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    MT5ForConditionalGeneration,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer, # Added Trainer import
    TrainingArguments
)
import evaluate
import numpy as np

In [ ]:
# Load a summarization dataset that does not rely on remote code (e.g., 'xsum')
dataset = load_dataset("xsum", trust_remote_code=False)

# Train-validation split check
print(dataset)

In [ ]:
dataset

In [ ]:

model_name = "google/mt5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = MT5ForConditionalGeneration.from_pretrained(model_name)

In [ ]:
max_input_length = 512
max_target_length = 128

def preprocess(example):
    input_text = "summarize: " + example["document"]

    model_inputs = tokenizer(
        input_text,
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        example["summary"],
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )

    # Replace padding token id's of labels by -100
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
def preprocess(example):
    input_texts = ["summarize: " + doc for doc in example["document"]]

    model_inputs = tokenizer(
        input_texts,
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        example["summary"],
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )

    # Replace padding token id's of labels by -100
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(
    preprocess,
    batched=True,
    remove_columns=dataset["train"].column_names
)

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [ ]:
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 with pad_token_id
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    return {k: round(v, 4) for k, v in result.items()}

In [ ]:
from transformers import TrainingArguments
import torch

training_args = TrainingArguments(
    output_dir="./mt5_xlsum",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    # predict_with_generate=True, # Temporarily removed due to unexpected TypeError
    logging_dir="./logs",
    logging_steps=100,
    fp16=torch.cuda.is_available()
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    # tokenizer=tokenizer, # Removed this line
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("./mt5_summarizer")
tokenizer.save_pretrained("./mt5_summarizer")

In [ ]:
from transformers import Trainer

# This block is added to ensure 'trainer' is defined and the model is trained
# before saving, addressing the 'NameError' from previous execution.
# This will re-run the training process if this cell is executed.
# Please ensure all variables used here (model, training_args, tokenized_dataset, data_collator, compute_metrics)
# are defined from previous cell executions.

print("Initializing Trainer and commencing training (if not already done) before saving the model.")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)
trainer.train()

trainer.save_model("./mt5_summarizer")
tokenizer.save_pretrained("./mt5_summarizer")

In [ ]:
def generate_summary(text):
    input_text = "summarize: " + text

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=100,
        num_beams=4,
        length_penalty=1.0,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# Example
sample_text = dataset["validation"][0]["text"]
print("Original:\n", sample_text)

print("\nSummary:\n", generate_summary(sample_text))